
#### 网页版大模型对话系统 - 完整起步代码
#### 使用 FastAPI + LangChain + 千问大模型

pip install fastapi uvicorn websockets langchain-community dashscope

In [6]:
from fastapi import APIRouter, WebSocket, WebSocketDisconnect, HTTPException

In [3]:
@tool
def get_current_time() -> str:
    """获取当前时间"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def calculate(expression: str) -> str:
    """
    计算数学表达式
    
    Args:
        expression: 数学表达式，例如 "2+2" 或 "10*5"
    """
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"


@tool
def search_web(query: str) -> str:
    """
    搜索网络信息（模拟）
    
    Args:
        query: 搜索关键词
    """
    # 这里可以集成真实的搜索API
    return f"关于'{query}'的搜索结果：这是一个模拟的搜索结果。实际应用中可以集成真实的搜索引擎。"


In [4]:
class ConversationManager:
    """对话管理器"""
    
    def __init__(self):
        self.conversations: Dict[str, List] = {}
        self.llm = ChatTongyi(
            model="qwen-plus",
            temperature=0.7,
            streaming=True,  # 启用流式输出
        )
        self.agent = self._create_agent()
    
    def _create_agent(self):
        """创建Agent"""
        tools = [get_current_time, calculate, search_web]
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", """你是一个智能助手，可以帮助用户完成各种任务。

你拥有以下工具：
1. get_current_time: 获取当前时间
2. calculate: 计算数学表达式
3. search_web: 搜索网络信息

请根据用户的需求选择合适的工具，用友好、专业的方式回答问题。"""),
            ("placeholder", "{chat_history}"),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}"),
        ])
        
        agent = create_tool_calling_agent(self.llm, tools, prompt)
        
        return AgentExecutor(
            agent=agent,
            tools=tools,
            verbose=True,
            handle_parsing_errors=True,
            max_iterations=3,
        )
    
    def get_history(self, session_id: str) -> List:
        """获取对话历史"""
        if session_id not in self.conversations:
            self.conversations[session_id] = []
        return self.conversations[session_id]
    
    def add_message(self, session_id: str, role: str, content: str):
        """添加消息到历史"""
        history = self.get_history(session_id)
        if role == "user":
            history.append(HumanMessage(content=content))
        else:
            history.append(AIMessage(content=content))
    
    async def chat_stream(self, session_id: str, message: str):
        """流式对话"""
        history = self.get_history(session_id)
        
        # 准备输入
        chat_history = history[-6:] if len(history) > 6 else history  # 只保留最近3轮对话
        
        try:
            # 调用Agent
            result = await asyncio.to_thread(
                self.agent.invoke,
                {
                    "input": message,
                    "chat_history": chat_history
                }
            )
            
            response = result['output']
            
            # 保存到历史
            self.add_message(session_id, "user", message)
            self.add_message(session_id, "assistant", response)
            
            # 流式返回（模拟打字效果）
            for i in range(0, len(response), 3):
                chunk = response[i:i+3]
                yield chunk
                await asyncio.sleep(0.03)  # 控制打字速度
                
        except Exception as e:
            error_msg = f"抱歉，处理您的请求时出现错误: {str(e)}"
            yield error_msg

In [ ]:
# 创建全局对话管理器
conversation_manager = ConversationManager()